In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

def load_project_env() -> Path:
    """Load the first .env found from the current folder up to the workspace root."""
    for candidate_dir in [Path.cwd(), *Path.cwd().resolve().parents]:
        candidate = candidate_dir / ".env"
        if candidate.exists():
            load_dotenv(candidate, override=True)
            return candidate
    raise FileNotFoundError("No se encontró el archivo .env")

env_file = load_project_env()
print(f"Loaded environment from: {env_file}")

# Variables cruciales para LangSmith

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = str(os.getenv("POSTGRESQL_MCP_LANGSMITH") or None)
os.environ["LANGSMITH_PROJECT"] = "Postresql MCP"

os.environ["NVIDIA_API_KEY"] = str(os.getenv("NVIDIA_API_KEY") or None)
os.environ["ORCHESTRATOR_API_KEY_LOCAL"] = str(os.getenv("ORCHESTRATOR_API_KEY_LOCAL") or None)
os.environ["ORCHESTRATOR_BASE_URL_LOCAL"] = str(os.getenv("ORCHESTRATOR_BASE_URL_LOCAL") or None)


from typing import Annotated, TypedDict
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.graph.message import add_messages
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI

Loaded environment from: /home/santi/Documentos/LangGraph/.env


In [3]:
system_env = dict(os.environ)

pg_user = os.getenv("PG_USER")
pg_password = os.getenv("PG_PASSWORD")
pg_host = os.getenv("PG_HOST")
pg_port = os.getenv("PG_PORT") or "5433"
pg_database = os.getenv("PG_DATABASE")

# MCP Client Configuration
client = MultiServerMCPClient(
    {
        "postgresql": {
            "command": "npx",
            "args": [
                "-y",
                "@modelcontextprotocol/server-postgres",
                f"postgresql://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_database}",
            ],
            "transport": "stdio",
            "env": system_env,
        }
    }
)

In [4]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [5]:
llm = ChatOpenAI(
    model="z-ai/glm-5.1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1", # NVIDIA's API URL
    temperature=0.0,
)

In [6]:
from langgraph.prebuilt import tools_condition
from langchain_core.tracers.context import tracing_v2_enabled


def handle_tool_error(error: Exception) -> str:
    return (
        "The database tool failed while executing a query. "
        f"Error detail: {error}. "
        "Do not assume table or column names. Reinspect the schema or metadata, "
        "then retry with a validated query."
    )


async def run_agent():
    async with client.session("postgresql") as mcp_session:
        tools = await load_mcp_tools(mcp_session)
        agent = llm.bind_tools(tools)

        # Call de Agent
        def call_agent(state: State):
            # Get messages from state
            messages = state["messages"]

            # Define Prompt
            agent_prompt = ("system", 
                "You are a expert assistant connected to the SEDICI database through MCP."
                "STRICT RULES: \n"
                "1. NEVER guess or asume the name of the tables, columns or any other database structure."
                "2. Your first step ALWAYS has to be use the tools to explore the database schemas available and get the information needed to answer the user's question (views) ."
                "3. Once you have the tables that seems relevant, inspect the structure (views, types of the columns, etc) before trying to answer the user's question."
                "4. If the query fails, read the error message, understand what went wrong, and use the tools again to get the correct information or fix the query before trying to answer the user's question again."

                "Answer the question based on the provided messages and using the tools when necessary."
            )   

            final_prompt = [agent_prompt] + messages
            # Answer
            response = agent.invoke(final_prompt)

            return {"messages": response}

        # Nodes
        workflow = StateGraph(State)
        workflow.add_node("agent", call_agent)
        workflow.add_node("tools_node", ToolNode(tools, handle_tool_errors=handle_tool_error))

        # Edges
        workflow.add_edge(START, "agent")

        workflow.add_conditional_edges(
            "agent", tools_condition,
            {
                "tools": "tools_node",
                END: END,
            },
        )
        workflow.add_edge("tools_node", "agent")

        # Compile graph
        app = workflow.compile(checkpointer=MemorySaver())

        # Execution
        inputs: State = {
            "messages": [
                ("user", "Calculate the distribution of authors based on the number of words in their names in the SEDICI database. I need a report showing the count of authors grouped by their word count (e.g., how many authors have 2-word names, 3-word names, etc.)")
            ]
        }
        config = {"configurable": {"thread_id": "1"}}

        with tracing_v2_enabled():
            result = await app.ainvoke(inputs, config=config)
    
        print(result["messages"][-1]) 

await run_agent()

content='Now I have all the data needed. Here\'s the complete report:\n\n---\n\n## 📊 Distribution of Authors by Word Count in Their Names — SEDICI Database\n\n**Source:** `mcp_dspace.vw_metadata` — field `sedici.creator.person`  \n**Total distinct authors:** 153,483\n\n| Word Count | # of Authors | % of Total | Description |\n|:----------:|:------------:|:----------:|:------------|\n| 1 | 287 | 0.19% | Single-word names (e.g., mononyms, or data quality issues like `",,,,,"`, `"6"`, `"Acha"`) |\n| **2** | **70,966** | **46.24%** | **Most common** — typical "Surname, Firstname" format (e.g., `Salvo, Nahuel`, `Molina, Víctor`) |\n| **3** | **63,658** | **41.48%** | **Second most common** — "Surname, Firstname Middlename" (e.g., `Vila, Gabriela Soledad`, `Lio, Vanesa Gisela`) |\n| 4 | 15,257 | 9.94% | Longer names (e.g., `Carballeda, Alfredo Juan Manuel`) |\n| 5 | 2,503 | 1.63% | |\n| 6 | 437 | 0.28% | |\n| 7 | 139 | 0.09% | |\n| 8 | 90 | 0.06% | |\n| 9 | 30 | 0.02% | |\n| 10 | 18 | 0.01% 

# Agent Output from Langsmith:

## 📊 Distribution of Authors by Word Count in Their Names — SEDICI Database

**Source:** `mcp_dspace.vw_metadata` — field `sedici.creator.person`

**Total distinct authors:** 153,483

| Word Count | # of Authors | % of Total | Description |
| :---: | :---: | :---: | :--- |
| 1 | 287 | 0.19% | Single-word names (e.g., mononyms, or data quality issues like `,,,,,`, `6`, `Acha`) |
| **2** | **70,966** | **46.24%** | **Most common** — typical "Surname, Firstname" format (e.g., `Salvo, Nahuel`, `Molina, Víctor`) |
| **3** | **63,658** | **41.48%** | **Second most common** — "Surname, Firstname Middlename" (e.g., `Vila, Gabriela Soledad`, `Lio, Vanesa Gisela`) |
| 4 | 15,257 | 9.94% | Longer names (e.g., `Carballeda, Alfredo Juan Manuel`) |
| 5 | 2,503 | 1.63% | |
| 6 | 437 | 0.28% | |
| 7 | 139 | 0.09% | |
| 8 | 90 | 0.06% | |
| 9 | 30 | 0.02% | |
| 10 | 18 | 0.01% | |
| 11 | 20 | 0.01% | |
| 12 | 11 | 0.01% | |
| 13–25 | 30 | 0.02% | |
| 26–64 | 8 | ~0.00% | |
| 75–325 | 7 | ~0.00% | ⚠️ Data quality issues |

### Key Findings

1. **~87.7% of all distinct authors have 2- or 3-word names**, which aligns with the typical Latin American naming convention: *"Surname, Firstname"* (2 words) or *"Surname, Firstname Middlename"* (3 words).
2. **2-word names are the most common** (46.24%), closely followed by 3-word names (41.48%).
3. **Names with 4–5 words** account for another ~11.6%, representing authors with longer compound surnames or multiple given names.
4. **Names with 6+ words** are extremely rare (< 0.5%) and largely represent **data quality issues** — inspection reveals these are typically:
   - **Multiple authors concatenated** in a single field (e.g., semicolon-separated or pipe-separated lists of several people)
   - **Affiliations/roles embedded** in the name field (e.g., `Lic. María Natalia García; Lic. Eugenia Ruiz; ...`)
   - **Full institutional descriptions** mistakenly entered as author names
5. **1-word "names"** (287 entries, 0.19%) are also mostly data quality issues — they include mononyms (`Acha`), missing-comma entries (`Acebal,Carolina`), and even numeric garbage (`6`, `99`).

### Recommendation
Entries with 6+ words (and many 1-word entries) should be reviewed for data normalization, as they likely represent parsing errors or bulk author lists that were not properly split into individual author records.